# AI-Assisted Student Performance Analysis and Prediction
**Prepared by:** Yuvraj Singh Baghel  
**Internship:** AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026  
**Project type:** Beginner-level data analysis and machine learning demonstration

## Project overview
This notebook explores a small **synthetic** student dataset, visualises patterns in study habits and attendance, and trains a simple machine-learning model to estimate whether a student may pass. The data is generated inside the notebook for demonstration; it is not real student data.

## Objectives
- Create and inspect a sample dataset.
- Clean and summarise the data using pandas.
- Visualise attendance, study hours, and marks.
- Train and evaluate a basic classification model.
- Explain limitations and possible improvements.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Create a sample dataset

In [ ]:
# Reproducible synthetic data for a simple classroom project
rng = np.random.default_rng(42)
n = 250

study_hours = rng.uniform(0.5, 6.0, n).round(1)
attendance = rng.integers(45, 101, n)
assignments = rng.integers(0, 11, n)

# Marks are influenced by study time, attendance and assignment completion,
# with some random variation so the data is not perfectly predictable.
noise = rng.normal(0, 8, n)
marks = (28 + study_hours * 6 + attendance * 0.25 + assignments * 1.2 + noise)
marks = np.clip(marks, 0, 100).round(1)
passed = (marks >= 50).astype(int)

df = pd.DataFrame({
    "study_hours_per_day": study_hours,
    "attendance_percent": attendance,
    "assignments_completed": assignments,
    "marks": marks,
    "passed": passed
})

df.head()


## 3. Inspect and prepare the data

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn types and non-null values:")
print(df.info())
print("\\nSummary statistics:")
display(df.describe())
print("\\nMissing values by column:")
print(df.isnull().sum())


In [ ]:
# Remove duplicate rows if any and fill missing numeric values if they occur.
df = df.drop_duplicates().copy()
for column in df.select_dtypes(include="number").columns:
    df[column] = df[column].fillna(df[column].median())

print("Rows after basic cleaning:", len(df))


## 4. Exploratory data analysis

In [ ]:
# Average marks by pass/fail group
group_summary = df.groupby("passed")[["study_hours_per_day", "attendance_percent", "marks"]].mean()
display(group_summary)


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["study_hours_per_day"], df["marks"], alpha=0.65)
plt.xlabel("Study hours per day")
plt.ylabel("Marks")
plt.title("Study time and marks (synthetic data)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
df["attendance_percent"].plot(kind="hist", bins=10, edgecolor="black")
plt.xlabel("Attendance (%)")
plt.title("Distribution of attendance")
plt.tight_layout()
plt.show()


## 5. Build a simple prediction model

In [ ]:
# We exclude marks from the features because marks are used to define the target.
features = ["study_hours_per_day", "attendance_percent", "assignments_completed"]
X = df[features]
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("\\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))


## 6. Try a sample prediction

In [ ]:
sample_student = pd.DataFrame([{
    "study_hours_per_day": 3.0,
    "attendance_percent": 80,
    "assignments_completed": 7
}])

prediction = model.predict(sample_student)[0]
print("Predicted class:", "Likely to pass" if prediction == 1 else "May need additional support")
print("Note: This is only a demonstration on synthetic data, not a real academic assessment.")


## 7. Findings and conclusion
In this sample, marks are designed to be influenced by study hours, attendance, assignments and random variation. The plots help inspect these relationships, while the classifier demonstrates a basic AI/ML workflow.

**Important limitations**
- The dataset is synthetic and small; results do not represent real students.
- Model accuracy can change with a different random sample or split.
- Study habits and attendance alone cannot explain every student's performance.
- This model should not be used to make real educational decisions.

## 8. Future scope
The project could be improved by using a properly permissioned real dataset, comparing more models, validating results with cross-validation, and documenting fairness and privacy safeguards.
